In [ ]:
!pip -q install google-genai

In [ ]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

print("API Key is ready!")

In [ ]:
%cd /content
!git clone -b main https://github.com/layan-sa1/glare-issue-discovery.git
%cd /content/glare-issue-discovery
!pip install -q -r requirements.txt

In [ ]:
from glare_issues.llm import GeminiLLM

llm = GeminiLLM(
    cache_path="/content/member5_cache.jsonl",
    model="gemini-3.1-flash-lite"
)

print("GeminiLLM is ready!")

In [ ]:
import pandas as pd

df = pd.read_csv("/content/01_preprocessed.csv")

print("Number of reviews:", len(df))
print("Columns:", df.columns.tolist())

df.head()

In [ ]:
import pandas as pd
import time
import os

progress_path = "/content/member5_filter_progress_latest.csv"
output_path = "/content/02_problem_reviews.csv"

# Start from the preprocessed dataset or resume saved progress
if os.path.exists(progress_path):
    progress = pd.read_csv(progress_path)

    if (
        len(progress) != len(df)
        or not progress["uid"].equals(df["uid"])
    ):
        raise ValueError("Progress file does not match preprocessed data")

    print("Resuming saved progress")

else:
    progress = df.copy()
    progress["is_problem"] = pd.NA
    progress.to_csv(progress_path, index=False)

    print("Starting classification from preprocessed data")

# Find only unclassified reviews
pending = progress.index[progress["is_problem"].isna()].tolist()

print("Total:", len(progress))
print("Already classified:", progress["is_problem"].notna().sum())
print("Remaining:", len(pending))

assert len(progress) == 30000
assert progress["is_problem"].dropna().isin([0, 1]).all()

batch_size = 20

for start in range(0, len(pending), batch_size):
    indices = pending[start:start + batch_size]
    texts = progress.loc[indices, "text"].fillna("").astype(str).tolist()

    try:
        labels = llm.classify_problem(texts, batch_size=20)

        if len(labels) != len(indices):
            raise ValueError("Labels count mismatch")

        progress.loc[indices, "is_problem"] = labels

        # Save progress after every batch
        progress.to_csv(progress_path, index=False)

        done = progress["is_problem"].notna().sum()
        print(f"Classified: {done}/30000")

        time.sleep(5)

    except Exception as e:
        progress.to_csv(progress_path, index=False)
        print("Stopped:", str(e)[:500])
        print("Progress saved.")
        break

remaining = progress["is_problem"].isna().sum()

if remaining == 0:
    progress["is_problem"] = progress["is_problem"].astype(int)

    problem_reviews = progress[progress["is_problem"] == 1].copy()
    problem_reviews.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print("Stage 1 completed!")
    print("Problem reviews:", len(problem_reviews))
    print("Output:", output_path)

else:
    print("Remaining reviews:", remaining)

In [ ]:
import os
from google.colab import files

progress_path = "/content/member5_filter_progress_latest.csv"
output_path = "/content/02_problem_reviews.csv"

if os.path.exists(progress_path):
    files.download(progress_path)

if os.path.exists(output_path):
    files.download(output_path)
else:
    print("Problem reviews file is not ready yet.")